# 03 - Broadcast Joins y Window Functions

### Broadcast Hash Join
Al unir una tabla grande con una dimensional pequeña (equipos), `F.broadcast()` replica la tabla pequeña en todos los ejecutores, **eliminando el Shuffle por red**.


In [ ]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA, calcular_imc
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = get_spark_session("03_Transformaciones")
df_dep = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")
df_eq = spark.read.option("header", "true").option("inferSchema", "true").csv("../data/raw/equipos.csv")

df_join = df_dep.transform(calcular_imc).join(F.broadcast(df_eq), on="equipo_id", how="inner")
df_join.select("nombre", "pais", "imc").show(5)


### Window Functions
Cálculo de rankings dentro de cada partición sin colapsar registros:


In [ ]:
ventana = Window.partitionBy("pais").orderBy(F.col("altura").desc())
df_rank = df_join.filter(F.col("altura").isNotNull()).withColumn("rank_altura", F.dense_rank().over(ventana))
df_rank.select("pais", "nombre", "altura", "rank_altura").show(10)
